# Extract features with ECGFounder

In [1]:
import os
import sys
import torch
import numpy as np

BASE_DIR = os.path.dirname(os.path.abspath('.'))
sys.path.append(BASE_DIR)

### Load the datasets & extract the features

In [29]:
from datautils import create_dataloader

ptbxl_loader = create_dataloader(# f"{BASE_DIR}/data/cinc/ptbxl/train.h5",
                                 # f"{BASE_DIR}/data/cinc/ptbxl/val.h5",
                                 f"{BASE_DIR}/data/cinc/ptbxl/test.h5",
                                 task="diagnostic_class", batch_size=32)
print("Number of batches [32] in PTB-XL loader:", len(ptbxl_loader))

samitrop_loader = create_dataloader(f"{BASE_DIR}/data/cinc/samitrop/all_data.h5",
                                    task="normal_ecg", batch_size=32)
print("Number of batches [32] in SamiTrop loader:", len(samitrop_loader))

Number of batches [32] in PTB-XL loader: 69
Number of batches [32] in SamiTrop loader: 51


In [30]:
ptbxl_labels = []
for _, labels in ptbxl_loader:
    ptbxl_labels.append(labels.numpy())
ptbxl_labels = np.concatenate(ptbxl_labels)

samitrop_labels = []
for _, labels in samitrop_loader:
    samitrop_labels.append(labels.numpy())
samitrop_labels = np.concatenate(samitrop_labels)

**Extract the features**

In [3]:
from models import ft_12lead_ECGFounder
from utils import set_seed


device = torch.device("cuda:7" if torch.cuda.is_available() else "cpu")
set_seed(42, deterministic=True)

ecgfounder = ft_12lead_ECGFounder(pth=f"{BASE_DIR}/.checkpoints/12_lead_ECGFounder.pth",
                                  n_classes=2, linear_prob=True, device=device)
ecgfounder.return_features = True

Seed set to 42 (deterministic=True)


/home/rafaelsilva/models/cnn-ecg-features/models/finetune_model.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(pth, map_location=device)


In [4]:
samitrop_features = []

ecgfounder.to(device)
ecgfounder.eval()

with torch.no_grad():

    for batch in samitrop_loader:
        x, y = batch[0].to(device), batch[1].to(device)

        _, deep_feat = ecgfounder(x)

        samitrop_features.append(deep_feat.detach().cpu())

ecgfounder.cpu()

samitrop_features = torch.concat(samitrop_features).numpy()
samitrop_features.shape

(1631, 1024)

In [ ]:
# np.save("samitrop_features.npy", samitrop_features)

In [ ]:
ptbxl_features = []

ecgfounder.to(device)
ecgfounder.eval()

with torch.no_grad():

    for batch in ptbxl_loader:
        x, y = batch[0].to(device), batch[1].to(device)

        _, deep_feat = ecgfounder(x)

        ptbxl_features.append(deep_feat.detach().cpu())

ecgfounder.cpu()

ptbxl_features = torch.concat(ptbxl_features).numpy()
ptbxl_features.shape

(2198, 1024)

In [ ]:
# np.save("ptbxl_test_features.npy", ptbxl_features)

### Embeddings distribution analysis

**UMAP Algorithm** - The major ones are as follows:

`n_neighbors`: This determines the number of neighboring points used in local approximations of manifold structure. Larger values will result in more global structure being preserved at the loss of detailed local structure. In general this parameter should often be in the range 5 to 50, with a choice of 10 to 15 being a sensible default.

`min_dist`: This controls how tightly the embedding is allowed compress points together. Larger values ensure embedded points are more evenly distributed, while smaller values allow the algorithm to optimise more accurately with regard to local structure. Sensible values are in the range 0.001 to 0.5, with 0.1 being a reasonable default.

`metric`: This determines the choice of metric used to measure distance in the input space. A wide variety of metrics are already coded, and a user defined function can be passed as long as it has been JITd by numba.

In [2]:
import umap

/home/rafaelsilva/models/cnn-ecg-features/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load the features from the .npy files
ptbxl_features = np.load("ptbxl_test_features.npy")
print("PTB-XL features shape:", ptbxl_features.shape)

samitrop_features = np.load("samitrop_features.npy")
print("SamiTrop features shape:", samitrop_features.shape)

all_features = np.concatenate((ptbxl_features, samitrop_features), axis=0)
print("All features shape:", all_features.shape)

# Create labels for the combined dataset (0 = PTB-XL, 1 = SamiTrop)
feat_dataset = np.concatenate((np.zeros(ptbxl_features.shape[0]), np.ones(samitrop_features.shape[0])))

PTB-XL features shape: (2198, 1024)
SamiTrop features shape: (1631, 1024)
All features shape: (3829, 1024)
Feature dataset labels shape: (3829,)


In [ ]:
# Perform UMAP dimensionality reduction (default parameters)
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
umap_features = reducer.fit_transform(all_features)

/home/rafaelsilva/models/cnn-ecg-features/env/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/home/rafaelsilva/models/cnn-ecg-features/env/lib/python3.10/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


In [3]:
# np.savez_compressed("umap_features.npz", umap_features=umap_features, labels=feat_dataset)
with np.load("umap_features.npz") as data:
    umap_features = data["embedding"]
    feat_dataset = data["labels"]

pip install -U kaleido

In [4]:
import plotly.express as ex

In [9]:
_feat_dataset = np.where(feat_dataset == 0, "PTB-XL", "SamiTrop")

In [67]:
fig = ex.scatter(x=umap_features[:,0], y=umap_features[:,1], color=_feat_dataset)

fig.update_layout(title="UMAP Projection of ECGFounder Features",
                  xaxis_title="UMAP Dimension 1",
                  yaxis_title="UMAP Dimension 2",
                  title_y=0.97,
                  legend=dict(title="Dataset",orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=.5),
                  margin=dict(l=20, r=20, t=70, b=20),
                  height=550, width=600)

fig.show()

In [37]:
_ptbxl_labels = np.where(ptbxl_labels[:,0] == 0, "ptbxl+abnormal", "ptbxl+normal")
_samitrop_labels = np.where(samitrop_labels == 0, "samitrop+abnormal", "samitrop+normal")
_feat_labels = np.concatenate((_ptbxl_labels, _samitrop_labels), axis=0)

In [63]:
fig = ex.scatter(x=umap_features[:,0], y=umap_features[:,1], color=_feat_labels)

fig.update_layout(title="UMAP Projection of ECGFounder Features",
                  xaxis_title="UMAP Dimension 1",
                  yaxis_title="UMAP Dimension 2",
                  title_y=0.97,
                  legend=dict(title="Class",orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=.5),
                  margin=dict(l=20, r=20, t=70, b=20),
                  height=550, width=600)

fig.show()